In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from occhio.benchmark.configs import BenchmarkDistributionName

In [ ]:
df = pd.read_parquet("data/standard_total/results.parquet")
flat = df.reset_index()

In [ ]:
# Best F1 and MCC per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best_f1 = group.loc[group["f1_score"].idxmax()]
    best_mcc = group.loc[group["mcc"].idxmax()]
    print(
        f"{bench:25s} "
        f"F1={best_f1['f1_score']:.4f} (l1={best_f1['l1_coefficient']})  "
        f"MCC={best_mcc['mcc']:.4f} (l1={best_mcc['l1_coefficient']})"
    )

In [ ]:
# Full results table sorted by F1
flat.sort_values("f1_score", ascending=False)[
    [
        "benchmark",
        "l1_coefficient",
        "sae_l0",
        "true_l0",
        "precision",
        "recall",
        "f1_score",
        "mcc",
        "explained_variance",
    ]
].head(12)

## F1 & MCC vs l1_coefficient across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "l1_coefficient", "sae_l0"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
).sort_values(["benchmark", "metric", "sae_l0"])

fig = px.line(
    melted,
    x="sae_l0",
    y="score",
    color="benchmark",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    markers=True,
    labels={
        "sae_l0": "SAE L0",
        "score": "Score",
        "benchmark": "Benchmark",
    },
    title="F1 & MCC vs SAE L0 by Distribution",
    height=500,
    width=1100,
)
fig.show()

## Precision & Recall vs l1_coefficient across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "l1_coefficient", "sae_l0"],
    value_vars=["precision", "recall"],
    var_name="metric",
    value_name="score",
).sort_values(["benchmark", "metric", "sae_l0"])

fig = px.line(
    melted,
    x="sae_l0",
    y="score",
    color="benchmark",
    facet_col="metric",
    category_orders={"metric": ["precision", "recall"]},
    markers=True,
    labels={
        "sae_l0": "SAE L0",
        "score": "Score",
        "benchmark": "Benchmark",
    },
    title="Precision & Recall vs SAE L0 by Distribution",
    height=500,
    width=1100,
)
fig.show()

## F1 & MCC heatmap: benchmark × l1_coefficient

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["F1 Score", "MCC"],
)

for i, metric in enumerate(["f1_score", "mcc"]):
    pivot = flat.pivot_table(values=metric, index="benchmark", columns="l1_coefficient")
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=pivot.index.tolist(),
            colorscale="Viridis",
            showscale=(i == 1),
            text=pivot.values.round(3),
            texttemplate="%{text}",
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=400,
    width=1100,
    title_text="Benchmark × L1 Coefficient",
)
fig.update_xaxes(title_text="l1_coefficient")
fig.show()

## sae_l0 vs true_l0 across distributions

In [ ]:
fig = px.scatter(
    flat,
    x="true_l0",
    y="sae_l0",
    color="benchmark",
    symbol="l1_coefficient",
    hover_data=["f1_score", "mcc"],
    labels={
        "true_l0": "True L0",
        "sae_l0": "SAE L0",
        "benchmark": "Benchmark",
        "l1_coefficient": "L1 Coeff",
    },
    title="SAE L0 vs True L0 (each point = one config)",
    height=500,
    width=800,
)
max_val = max(flat["true_l0"].max(), flat["sae_l0"].max())
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(dash="dash", color="gray"),
)
fig.show()